# ChessMind Arena — Chess Policy Model Training

This notebook trains the first chess policy network for ChessMind Arena.

The model learns:

    Chess Position → Strong Candidate Move

The training data comes from the prepared Hugging Face dataset:

    chessmind-arena-policy-dataset

Each example contains:

- FEN
- Stockfish principal-variation first move
- Stockfish evaluation
- search depth
- node count

The first model is a CNN-based policy network.

The model is not intended to replace the LLM.

Instead, it will eventually be used as:

    Chess Position
          ↓
    Chess Policy Network
          ↓
    Top-K Candidate Moves
          ↓
    LLM Player
          ↓
    Selected Move

This notebook establishes the first supervised-learning baseline.

In [1]:
!pip install -q datasets python-chess torch pandas numpy tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 54.0 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [2]:
import chess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42
BATCH_SIZE = 256
learning_rate = 1e-3
EPOCHS = 10

NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [5]:
import getpass
import os

os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face Token: ")

In [6]:
HF_DATASET_ID = "Mridul-Dixit/chessmind-arena-policy-dataset"

dataset = load_dataset(HF_DATASET_ID)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 10000
    })
})


In [7]:
sample = dataset["train"][0]

In [9]:
print("FEN:", sample["fen"])
print("Target move:", sample["target_move"])
print("Depth:", sample["depth"])
print("CP:", sample["cp"])
print("Mate:", sample["mate"])

FEN: 8/1R6/3p1p2/4bP1p/4k3/1R4P1/7r/3K4 w - -
Target move: b7b4
Depth: 65
CP: 0.0
Mate: None


In [10]:
board = chess.Board(sample["fen"])

print(board)

. . . . . . . .
. R . . . . . .
. . . p . p . .
. . . . b P . p
. . . . k . . .
. R . . . . P .
. . . . . . . r
. . . K . . . .


In [ ]:
print("\nTarget move:", sample["target_move"])
move = chess.Move.from_uci(sample["target_move"])
print("Is legal:", move in board.legal_moves)


Target move: b7b4
Is legal: True


In [12]:
PIECE_TO_CHANNEL = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,

    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}


def encode_board(fen):
    board = chess.Board(fen)

    encoded = np.zeros(
        (12, 8, 8),
        dtype=np.float32
    )

    for square, piece in board.piece_map().items():

        channel = PIECE_TO_CHANNEL[
            (piece.color, piece.piece_type)
        ]

        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)

        encoded[channel, row, col] = 1.0

    return encoded

In [13]:
encoded_board = encode_board(sample["fen"])

print("Encoded board shape:", encoded_board.shape)
print("Non zero entries:", np.count_nonzero(encoded_board))

Encoded board shape: (12, 8, 8)
Non zero entries: 11


In [14]:
NUM_SQUARES = 64
BASE_MOVE_CLASSES = 64 * 64

PROMOTION_PIECES = [
    chess.KNIGHT,
    chess.BISHOP,
    chess.ROOK,
    chess.QUEEN,
]

PROMOTION_OFFSET = BASE_MOVE_CLASSES

PROMOTION_TO_INDEX = {
    chess.KNIGHT: 0,
    chess.BISHOP: 1,
    chess.ROOK: 2,
    chess.QUEEN: 3,
}

NUM_MOVE_CLASSES = BASE_MOVE_CLASSES + (
    64 * 64 * len(PROMOTION_PIECES)
)


def encode_move(uci_move):
    move = chess.Move.from_uci(uci_move)

    from_square = move.from_square
    to_square = move.to_square

    base_index = from_square * 64 + to_square

    if move.promotion is None:
        return base_index

    promotion_index = PROMOTION_TO_INDEX[move.promotion]

    promotion_index = (
        promotion_index * 64 * 64
        + base_index
    )

    return PROMOTION_OFFSET + promotion_index


def decode_move(move_index):
    if move_index < BASE_MOVE_CLASSES:

        from_square = move_index // 64
        to_square = move_index % 64

        return chess.Move(
            from_square,
            to_square
        )

    promotion_index = move_index - PROMOTION_OFFSET

    promotion_type = promotion_index // (64 * 64)

    base_index = promotion_index % (64 * 64)

    from_square = base_index // 64
    to_square = base_index % 64

    promotion_piece = PROMOTION_PIECES[promotion_type]

    return chess.Move(
        from_square,
        to_square,
        promotion=promotion_piece
    )

In [15]:
test_moves = [
    "e2e4",
    "g1f3",
    "e7e8q",
    "a7a8n",
]

for move_uci in test_moves:

    index = encode_move(move_uci)
    decoded = decode_move(index)

    print(
        move_uci,
        "→",
        index,
        "→",
        decoded.uci()
    )

e2e4 → 796 → e2e4
g1f3 → 405 → g1f3
e7e8q → 19772 → e7e8q
a7a8n → 7224 → a7a8n


In [16]:
class ChessPolicyDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        row = self.dataset[index]

        board_tensor = encode_board(
            row["fen"]
        )

        target = encode_move(
            row["target_move"]
        )

        return (
            torch.tensor(
                board_tensor,
                dtype=torch.float32
            ),
            torch.tensor(
                target,
                dtype=torch.long
            )
        )

In [17]:
train_dataset = ChessPolicyDataset(
    dataset["train"]
)
val_dataset = ChessPolicyDataset(
    dataset["validation"]
)
test_dataset = ChessPolicyDataset(
    dataset["test"]
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 80000
Validation dataset size: 10000
Test dataset size: 10000


In [20]:
X, y = train_dataset[0]

print("Input shape:", X.shape)
print("Target:", y.item())
print("Target move:", dataset["train"][0]["target_move"])


Input shape: torch.Size([12, 8, 8])
Target: 3161
Target move: b7b4
